# Scale Dictionary Block-wise to Group-wise Conversion

This notebook converts block-wise scale parameters to group-wise by repeating each row 16 times.


In [ ]:
import torch
import numpy as np
from pathlib import Path
import shutil

In [3]:
# 设置文件路径
scale_dict_path = "/data/zihan/dev_block/scale.pt"
output_path = "/data/zihan/dev_block/scale_blockwise.pt"

# 加载原始权重文件
print(f"加载权重文件: {scale_dict_path}")
scale_dict = torch.load(scale_dict_path, map_location="cpu")

print(f"权重文件类型: {type(scale_dict)}")
if isinstance(scale_dict, dict):
    print(f"权重文件键的数量: {len(scale_dict)}")
    print("键列表:")
    for i, key in enumerate(scale_dict.keys()):
        print(f"  {i+1}. {key}")
else:
    print(f"权重文件不是字典，而是: {type(scale_dict)}")


加载权重文件: /data/zihan/dev_block/scale.pt
权重文件类型: <class 'dict'>
权重文件键的数量: 1520
键列表:
  1. transformer_blocks.0.norm1.linear.weight.scale.0
  2. transformer_blocks.0.norm1.linear.weight.zero
  3. transformer_blocks.0.norm1_context.linear.weight.scale.0
  4. transformer_blocks.0.norm1_context.linear.weight.zero
  5. transformer_blocks.0.attn.to_q.weight.scale.0
  6. transformer_blocks.0.attn.to_q.weight.scale.1
  7. transformer_blocks.0.attn.to_q.weight.zero
  8. transformer_blocks.0.attn.to_k.weight.scale.0
  9. transformer_blocks.0.attn.to_k.weight.scale.1
  10. transformer_blocks.0.attn.to_k.weight.zero
  11. transformer_blocks.0.attn.to_v.weight.scale.0
  12. transformer_blocks.0.attn.to_v.weight.scale.1
  13. transformer_blocks.0.attn.to_v.weight.zero
  14. transformer_blocks.0.attn.add_q_proj.weight.scale.0
  15. transformer_blocks.0.attn.add_q_proj.weight.scale.1
  16. transformer_blocks.0.attn.add_q_proj.weight.zero
  17. transformer_blocks.0.attn.add_k_proj.weight.scale.0
  18. tra

In [4]:
# 查看原始权重的形状信息
if isinstance(scale_dict, dict):
    print("=== 原始权重形状信息 ===")
    for key, value in scale_dict.items():
        if isinstance(value, torch.Tensor):
            print(f"{key}:")
            print(f"  形状: {value.shape}")
            print(f"  数据类型: {value.dtype}")
            print(f"  设备: {value.device}")
            print(f"  第一维大小: {value.shape[0] if len(value.shape) > 0 else 'N/A'}")
            print()
else:
    print("权重文件不是字典格式")


=== 原始权重形状信息 ===
transformer_blocks.0.norm1.linear.weight.scale.0:
  形状: torch.Size([288, 1, 48, 1])
  数据类型: torch.float32
  设备: cpu
  第一维大小: 288

transformer_blocks.0.norm1.linear.weight.zero:
  形状: torch.Size([])
  数据类型: torch.float32
  设备: cpu
  第一维大小: N/A

transformer_blocks.0.norm1_context.linear.weight.scale.0:
  形状: torch.Size([288, 1, 48, 1])
  数据类型: torch.float32
  设备: cpu
  第一维大小: 288

transformer_blocks.0.norm1_context.linear.weight.zero:
  形状: torch.Size([])
  数据类型: torch.float32
  设备: cpu
  第一维大小: N/A

transformer_blocks.0.attn.to_q.weight.scale.0:
  形状: torch.Size([1, 1, 1, 1])
  数据类型: torch.float32
  设备: cpu
  第一维大小: 1

transformer_blocks.0.attn.to_q.weight.scale.1:
  形状: torch.Size([192, 1, 192, 1])
  数据类型: torch.float32
  设备: cpu
  第一维大小: 192

transformer_blocks.0.attn.to_q.weight.zero:
  形状: torch.Size([])
  数据类型: torch.float32
  设备: cpu
  第一维大小: N/A

transformer_blocks.0.attn.to_k.weight.scale.0:
  形状: torch.Size([1, 1, 1, 1])
  数据类型: torch.float32
  设备: cpu
  第一维大小:

In [5]:
def convert_blockwise_to_groupwise(scale_dict, default_repeat_factor=16, norm_repeat_factor=64):
    """
    将block-wise的scale参数转换为group-wise参数
    通过将每一行重复repeat_factor次来实现
    只有第一维大小存在且不是1时才进行修改
    
    特殊规则：如果键名中包含"norm"，使用norm_repeat_factor（默认32），否则使用default_repeat_factor（默认16）
    
    Args:
        scale_dict: 原始权重字典
        default_repeat_factor: 默认重复因子，默认为16
        norm_repeat_factor: norm层的重复因子，默认为32
    
    Returns:
        转换后的权重字典
    """
    converted_dict = {}
    
    for key, value in scale_dict.items():
        # 判断是否是norm层，选择合适的重复因子
        if 'norm' in key.lower():
            repeat_factor = norm_repeat_factor
        else:
            repeat_factor = default_repeat_factor
        
        if isinstance(value, torch.Tensor):
            print(f"转换 {key}: {value.shape} -> ", end="")
            
            # 检查张量维度
            if len(value.shape) == 0:
                # 标量张量，直接复制
                converted_dict[key] = value.clone()
                print(f"{value.shape} (标量，保持不变)")
            elif len(value.shape) >= 1 and value.shape[0] == 1:
                # 第一维大小为1，直接复制
                converted_dict[key] = value.clone()
                print(f"{value.shape} (第一维为1，保持不变)")
            elif len(value.shape) >= 1 and value.shape[0] > 1:
                # 第一维大小存在且大于1，进行重复操作
                print(f"(factor={repeat_factor}) ", end="")
                
                if len(value.shape) == 1:
                    # 1D张量，将每个元素重复repeat_factor次
                    repeated_values = []
                    for val in value:
                        repeated_values.extend([val] * repeat_factor)
                    converted_dict[key] = torch.tensor(repeated_values, dtype=value.dtype)
                    print(f"{converted_dict[key].shape}")
                else:
                    # 多维张量，将第一维的每一行重复repeat_factor次
                    # 使用repeat_interleave来重复每一行
                    converted_dict[key] = value.repeat_interleave(repeat_factor, dim=0)
                    print(f"{converted_dict[key].shape}")
            else:
                # 其他情况，直接复制
                converted_dict[key] = value.clone()
                print(f"{value.shape} (其他情况，保持不变)")
        else:
            # 非张量值，直接复制
            converted_dict[key] = value
            print(f"{key}: 非张量值，保持不变")
    
    return converted_dict

def convert_blockwise_to_groupwise_norm(scale_dict, norm_repeat_factor=64):
    """
    将 block-wise 的 scale 参数转换为 group-wise 参数。
    修改点：仅对包含 "norm" 的键应用重复（使用 norm_repeat_factor=64），其他层保持不变。

    Args:
        scale_dict: 原始权重字典 (Tensor/非Tensor混合)
        default_repeat_factor: 保留参数（未使用），与旧接口兼容
        norm_repeat_factor: norm 层的重复因子（默认 64）

    Returns:
        转换后的权重字典
    """
    converted_dict = {}

    for key, value in scale_dict.items():
        if isinstance(value, torch.Tensor):
            print(f"转换 {key}: {tuple(value.shape)} -> ", end="")

            # 仅对 norm 层做重复，其它层直接保持不变
            if 'norm' in key.lower():
                repeat_factor = norm_repeat_factor
                # 标量/0D：不重复
                if value.ndim == 0:
                    converted_dict[key] = value.clone()
                    print(f"{tuple(value.shape)} (标量，保持不变)")
                # 1D：按元素重复 repeat_factor 次
                elif value.ndim == 1:
                    converted_dict[key] = value.repeat_interleave(repeat_factor)
                    print(f"{tuple(converted_dict[key].shape)} (1D 按元素重复 {repeat_factor})")
                else:
                    # 多维：在 dim=0 按行重复 repeat_factor 次
                    converted_dict[key] = value.repeat_interleave(repeat_factor, dim=0)
                    print(f"{tuple(converted_dict[key].shape)} (dim0 按行重复 {repeat_factor})")
            else:
                # 非 norm 层：完全保持不变
                converted_dict[key] = value.clone()
                print(f"{tuple(value.shape)} (非 norm，保持不变)")
        else:
            converted_dict[key] = value
            print(f"{key}: 非张量值，保持不变")

    return converted_dict

# 执行转换
print("=== 开始转换 ===")
print("规则: norm层使用factor=64, 其他层使用factor=16")
print()
converted_scale_dict = convert_blockwise_to_groupwise_norm(scale_dict, norm_repeat_factor=64)
print("\n转换完成！")


=== 开始转换 ===
规则: norm层使用factor=64, 其他层使用factor=16

转换 transformer_blocks.0.norm1.linear.weight.scale.0: (288, 1, 48, 1) -> (18432, 1, 48, 1) (dim0 按行重复 64)
转换 transformer_blocks.0.norm1.linear.weight.zero: () -> () (标量，保持不变)
转换 transformer_blocks.0.norm1_context.linear.weight.scale.0: (288, 1, 48, 1) -> (18432, 1, 48, 1) (dim0 按行重复 64)
转换 transformer_blocks.0.norm1_context.linear.weight.zero: () -> () (标量，保持不变)
转换 transformer_blocks.0.attn.to_q.weight.scale.0: (1, 1, 1, 1) -> (1, 1, 1, 1) (非 norm，保持不变)
转换 transformer_blocks.0.attn.to_q.weight.scale.1: (192, 1, 192, 1) -> (192, 1, 192, 1) (非 norm，保持不变)
转换 transformer_blocks.0.attn.to_q.weight.zero: () -> () (非 norm，保持不变)
转换 transformer_blocks.0.attn.to_k.weight.scale.0: (1, 1, 1, 1) -> (1, 1, 1, 1) (非 norm，保持不变)
转换 transformer_blocks.0.attn.to_k.weight.scale.1: (192, 1, 192, 1) -> (192, 1, 192, 1) (非 norm，保持不变)
转换 transformer_blocks.0.attn.to_k.weight.zero: () -> () (非 norm，保持不变)
转换 transformer_blocks.0.attn.to_v.weight.scale.0: (1, 1, 

In [6]:
# 验证转换结果
print("=== 转换后权重形状信息 ===")
for key, value in converted_scale_dict.items():
    if isinstance(value, torch.Tensor):
        print(f"{key}:")
        print(f"  形状: {value.shape}")
        print(f"  数据类型: {value.dtype}")
        print(f"  设备: {value.device}")
        print()

# 比较转换前后的差异
print("=== 转换前后对比 ===")
for key in scale_dict.keys():
    if isinstance(scale_dict[key], torch.Tensor) and isinstance(converted_scale_dict[key], torch.Tensor):
        original_shape = scale_dict[key].shape
        converted_shape = converted_scale_dict[key].shape
        
        print(f"{key}:")
        print(f"  原始形状: {original_shape}")
        print(f"  转换后形状: {converted_shape}")
        
        if len(original_shape) > 0 and len(converted_shape) > 0:
            expansion_factor = converted_shape[0] / original_shape[0]
            print(f"  第一维扩展倍数: {expansion_factor}")
        print()


=== 转换后权重形状信息 ===
transformer_blocks.0.norm1.linear.weight.scale.0:
  形状: torch.Size([18432, 1, 48, 1])
  数据类型: torch.float32
  设备: cpu

transformer_blocks.0.norm1.linear.weight.zero:
  形状: torch.Size([])
  数据类型: torch.float32
  设备: cpu

transformer_blocks.0.norm1_context.linear.weight.scale.0:
  形状: torch.Size([18432, 1, 48, 1])
  数据类型: torch.float32
  设备: cpu

transformer_blocks.0.norm1_context.linear.weight.zero:
  形状: torch.Size([])
  数据类型: torch.float32
  设备: cpu

transformer_blocks.0.attn.to_q.weight.scale.0:
  形状: torch.Size([1, 1, 1, 1])
  数据类型: torch.float32
  设备: cpu

transformer_blocks.0.attn.to_q.weight.scale.1:
  形状: torch.Size([192, 1, 192, 1])
  数据类型: torch.float32
  设备: cpu

transformer_blocks.0.attn.to_q.weight.zero:
  形状: torch.Size([])
  数据类型: torch.float32
  设备: cpu

transformer_blocks.0.attn.to_k.weight.scale.0:
  形状: torch.Size([1, 1, 1, 1])
  数据类型: torch.float32
  设备: cpu

transformer_blocks.0.attn.to_k.weight.scale.1:
  形状: torch.Size([192, 1, 192, 1])
  数据类型: t

In [7]:
# 保存转换后的权重文件
print(f"保存转换后的权重文件到: {output_path}")
torch.save(converted_scale_dict, output_path)

# 验证保存的文件
print("\n验证保存的文件...")
loaded_converted = torch.load(output_path, map_location="cpu")
print(f"保存的文件类型: {type(loaded_converted)}")
print(f"保存的文件键数量: {len(loaded_converted) if isinstance(loaded_converted, dict) else 'N/A'}")

# 检查保存的文件是否正确
if isinstance(loaded_converted, dict):
    print("\n保存文件的形状:")
    for key, value in loaded_converted.items():
        if isinstance(value, torch.Tensor):
            print(f"  {key}: {value.shape}")

print("\n转换和保存完成！")


保存转换后的权重文件到: /data/zihan/dev_block/scale_blockwise.pt

验证保存的文件...
保存的文件类型: <class 'dict'>
保存的文件键数量: 1520

保存文件的形状:
  transformer_blocks.0.norm1.linear.weight.scale.0: torch.Size([18432, 1, 48, 1])
  transformer_blocks.0.norm1.linear.weight.zero: torch.Size([])
  transformer_blocks.0.norm1_context.linear.weight.scale.0: torch.Size([18432, 1, 48, 1])
  transformer_blocks.0.norm1_context.linear.weight.zero: torch.Size([])
  transformer_blocks.0.attn.to_q.weight.scale.0: torch.Size([1, 1, 1, 1])
  transformer_blocks.0.attn.to_q.weight.scale.1: torch.Size([192, 1, 192, 1])
  transformer_blocks.0.attn.to_q.weight.zero: torch.Size([])
  transformer_blocks.0.attn.to_k.weight.scale.0: torch.Size([1, 1, 1, 1])
  transformer_blocks.0.attn.to_k.weight.scale.1: torch.Size([192, 1, 192, 1])
  transformer_blocks.0.attn.to_k.weight.zero: torch.Size([])
  transformer_blocks.0.attn.to_v.weight.scale.0: torch.Size([1, 1, 1, 1])
  transformer_blocks.0.attn.to_v.weight.scale.1: torch.Size([192, 1, 192, 1])